In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [10]:
from openai import OpenAI
import os

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://api.apipro.me/v1"
)

In [13]:
def ask(prompt: str, system: str = "You are a helpful assistant.", **kwargs) -> str:
    response = client.chat.completions.create(
        model="ollama-kimi-k2.5",
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": prompt}
        ],
        **kwargs
    )
    return response.choices[0].message.content


In [12]:
print(ask("Hello, who are you?"))

Hello! I'm an AI assistant, here to help answer your questions, have conversations, assist with writing, analysis, coding, or just chat about whatever's on your mind. How can I help you today?


In [14]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [15]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [16]:
len(documents)  # Check how many documents were read

72

In [18]:
documents[70]  # Preview the content of the first document

{'content': '# Summary and Closing Remarks\n\n<a href="https://www.youtube.com/watch?v=TW9M5VE8vpo&list=PL3MmuxUbc_hIB4fSqLy_0AfTjVLpgjV3R">\n  <img src="https://markdown-videos-api.jorgenkh.no/youtube/TW9M5VE8vpo">\n</a>\n\nBetween the previous video and this one, the project was polished\nfurther. Let\'s look at what changed and the final result.\n\n## Post-recording improvements\n\nSeveral improvements were made after recording:\n\n- PostgreSQL logging was finished (there were timestamp issues with\n  Docker clocks being out of sync with the host)\n- Grafana dashboard provisioning was automated with an init script\n  that creates the data source and loads the dashboard JSON\n- The README was polished with a generated banner image, better\n  formatting, grammar fixes, and clearer instructions\n- Code readability improvements across all files\n\n## Total cost\n\nThe entire project cost about $2 in OpenAI API calls:\n\n- Dataset generation: ~$0.50\n- Ground truth generation: ~$0.50\n- 

In [21]:
from minsearch import Index

index = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)
index.fit(documents)

In [22]:
question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question
)

search_results

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

In [23]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

--2026-06-22 22:16:43--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py’

rag_helper.py       100%[===================>]   2.08K  --.-KB/s    in 0.002s  

2026-06-22 22:16:44 (980 KB/s) - ‘rag_helper.py’ saved [2134/2134]



In [30]:
from rag_helper import RAGBase, INSTRUCTIONS, PROMPT_TEMPLATE
from dataclasses import dataclass

@dataclass
class RAGResult:
    answer: str
    input_tokens: int
    output_tokens: int


class RAGMarkdown(RAGBase):
    """
    RAGBase adapted for markdown docs with 'filename' and 'content' fields.
    Two overrides vs the FAQ version:
      - search     : no keyword filter, boost content over filename
      - build_context : format as "File: ...\n<content>" blocks
    Two modifications for token tracking:
      - llm  : returns the full response object (not just output_text)
      - rag  : returns a RAGResult(answer, input_tokens, output_tokens)
    """

    # ── Schema override 1: no course keyword filter ──────────────────────────
    def search(self, query, num_results=5):
        return self.index.search(
            query,
            num_results=num_results,
            boost_dict={"content": 1.0, "filename": 0.5},
        )

    # ── Schema override 2: filename/content → context ────────────────────────
    def build_context(self, search_results):
        lines = []
        for doc in search_results:
            lines.append(f"File: {doc['filename']}")
            lines.append(doc["content"])
            lines.append("")
        return "\n".join(lines).strip()

    # ── Token-aware llm: return full response ─────────────────────────────────
    def llm(self, prompt):
        messages = [
        {"role": "system", "content": self.instructions},  # "developer" → "system"
        {"role": "user",   "content": prompt},
    ]
        response = self.llm_client.chat.completions.create(
        model=self.model,
        messages=messages,
    )
        return response  # full object

    # ── Token-aware rag: return RAGResult ─────────────────────────────────────
    def rag(self, query):
        search_results = self.search(query)
        prompt   = self.build_prompt(query, search_results)
        response = self.llm(prompt)
        return RAGResult(
        answer=response.choices[0].message.content,
        input_tokens=response.usage.prompt_tokens,       # ← chat completions key
        output_tokens=response.usage.completion_tokens,  # ← chat completions key
    )

In [31]:
if __name__ == "__main__":
    client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://api.apipro.me/v1"
) 
    assistant = RAGMarkdown(
        index=index,
        llm_client=client,
        model="ollama-kimi-k2.5",  # course alias: gpt-5.4-mini
    )
 
    result = assistant.rag(
        "How does the agentic loop keep calling the model until it stops?"
    )
 
    print("ANSWER:")
    print(result.answer)
    print(f"\nInput (prompt) tokens : {result.input_tokens}")
    print(f"Output tokens         : {result.output_tokens}")

ANSWER:
The agentic loop uses a **`while True` loop** that continues until the model returns a response without any function calls.

Here is how it works:

1.  **Iteration**: Each iteration of the loop sends the current message history (including the user's question and any previous tool results) to the model.
2.  **Check for tool calls**: After receiving the response, the code checks if the output contains any `function_call` entries.
3.  **Execute and continue**: If there are function calls, the code executes them, appends the results back to the message history, sets a flag `has_function_calls = True`, and the loop continues to the next iteration so the model can see the results.
4.  **Exit condition**: If the response contains no function calls (only a final message), the flag remains `False`, and the loop breaks.

As shown in the code from the context:

```python
while True:
    has_function_calls = False
    response = openai_client.responses.create(...)
    
    for item in resp

In [32]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Number of chunks: {len(chunks)}")

Number of chunks: 295


In [33]:
# Create index from chunks (content as text field, filename as keyword field)
chunk_index = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)
chunk_index.fit(chunks)

In [34]:
# Create RAG assistant with chunk index
chunk_assistant = RAGMarkdown(
    index=chunk_index,
    llm_client=client,
    model="ollama-kimi-k2.5",
)

# Query with the chunk index
chunk_result = chunk_assistant.rag(
    "How does the agentic loop keep calling the model until it stops?"
)

print("CHUNKED VERSION ANSWER:")
print(chunk_result.answer)
print(f"\nInput (prompt) tokens (chunked)  : {chunk_result.input_tokens}")
print(f"Output tokens (chunked)          : {chunk_result.output_tokens}")

# Compare with Q3
print(f"\n--- COMPARISON ---")
print(f"Q3 (original documents) input tokens : {result.input_tokens}")
print(f"Q4 (chunked documents) input tokens  : {chunk_result.input_tokens}")
print(f"\nToken reduction: {result.input_tokens - chunk_result.input_tokens} tokens")
print(f"Percentage reduction: {((result.input_tokens - chunk_result.input_tokens) / result.input_tokens * 100):.1f}%")

CHUNKED VERSION ANSWER:
The agentic loop uses a `while True` loop with a boolean flag that checks whether the model requested any function calls in the current turn.

Here's how it works:

1. **Initialize a flag** (`has_function_calls`) to `False` at the start of each iteration
2. **Process the response**: Loop through the model's output items looking for function calls
3. **Set the flag**: If any item is a `function_call`, execute the tool and set `has_function_calls = True`
4. **Check the exit condition**: At the end of the iteration, if `has_function_calls` is still `False` (meaning the model returned only a message/answer with no tool requests), the loop breaks

As shown in the code:
```python
while True:
    has_function_calls = False
    
    # ... call model and process output ...
    
    for item in response.output:
        if item.type == "function_call":
            # execute tool...
            has_function_calls = True
    
    if has_function_calls == False:
        break

In [38]:
import json

# Define the search tool function
def search(query: str) -> str:
    """
    Search the course materials using the chunk index.
    
    Args:
        query: The search query to find relevant course content
        
    Returns:
        A formatted string with the search results
    """
    results = chunk_index.search(query, num_results=5)
    
    # Format results
    output_lines = []
    for i, doc in enumerate(results, 1):
        output_lines.append(f"Result {i} (from {doc['filename']}):")
        output_lines.append(doc['content'][:400])  # First 400 chars
        output_lines.append("")
    
    return "\n".join(output_lines) if output_lines else "No results found"

# Define the tool schema for the API
tools = [
    {
        "type": "function",
        "function": {
            "name": "search",
            "description": "Search the course materials using the chunk index.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query to find relevant course content"
                    }
                },
                "required": ["query"]
            }
        }
    }
]

# Agentic loop with search tool
def run_agent_with_search(question: str, max_iterations: int = 10) -> tuple[str, int]:
    """
    Run an agentic loop that can call search tool.
    Returns the final answer and number of search calls.
    """
    search_call_count = 0
    messages = [
        {
            "role": "system",
            "content": """You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."""
        },
        {
            "role": "user",
            "content": question
        }
    ]
    
    print(f"Question: {question}\n")
    print("=" * 80)
    
    for iteration in range(max_iterations):
        # Call the model
        response = client.chat.completions.create(
            model="ollama-kimi-k2.5",
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )
        
        # Check if model wants to use a tool
        if response.choices[0].finish_reason == "tool_calls":
            # Process tool calls
            assistant_message = response.choices[0].message
            for tool_call in assistant_message.tool_calls:
                if tool_call.function.name == "search":
                    search_call_count += 1
                    query = json.loads(tool_call.function.arguments)["query"]
                    print(f"\n🔍 Search call #{search_call_count}: '{query}'")
                    
                    # Execute search
                    search_result = search(query)
                    print(f"Results: {search_result[:200]}...")
                    
                    # Add assistant response with tool use
                    messages.append({"role": "assistant", "content": assistant_message.content, "tool_calls": assistant_message.tool_calls})
                    
                    # Add tool result
                    messages.append({
                        "role": "user",
                        "content": f"Tool result for search '{query}':\n{search_result}"
                    })
        else:
            # Model has finished with final answer
            final_answer = response.choices[0].message.content
            print("=" * 80)
            print(f"\n✅ Agent response:\n{final_answer}")
            print(f"\n📊 Total search calls: {search_call_count}")
            return final_answer, search_call_count
    
    print(f"\nReached max iterations ({max_iterations})")
    return None, search_call_count

# Run the agent
question = "How does the agentic loop work, and how is it different from plain RAG?"
answer, search_calls = run_agent_with_search(question)

Question: How does the agentic loop work, and how is it different from plain RAG?


🔍 Search call #1: 'agentic loop'
Results: Result 1 (from 01-agentic-rag/lessons/15-frameworks.md):
esult.

The `result` is a `LoopResult` with `all_messages` (the full
conversation), token counts, and `cost` (computed from token usage).

## C...

🔍 Search call #2: 'RAG retrieval augmented generation'
Results: Result 1 (from 03-orchestration/lessons/05-rag.md):
# Retrieval Augmented Generation

Video: [RAG Workflows](https://youtu.be/FhGZV173xrk)

AI Copilot solves the context problem for flow generation. B...

🔍 Search call #3: 'agent vs RAG difference'
Results: Result 1 (from 03-orchestration/lessons/07-multi-agent.md):
# Multi-Agent Systems

Video: [Agentic Workflows](https://youtu.be/7tvpR8EE0gs)

For complex tasks, you can design systems where multiple sp...

🔍 Search call #4: 'while loop tool calling agent'
Results: Result 1 (from 03-orchestration/lessons/07-multi-agent.md):
# Multi-Agent Systems

